# Final Validation Set Model Evaluation Pipeline

This notebook loads the frozen validation dataset split, queries the MLflow SQLite backend 
to fetch the highest-performing model pipeline artifact, and computes generalization metrics.

In [7]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Environment Configurations & Data Ingestion

In [8]:
# Define core path anchors relative to notebook location
EXPERIMENT_NAME = "customer-churn-optuna"

ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

In [9]:
# Ingest Data Partitions
VAL_FEATURES_PATH = "../data/processed/raw_features/X_val.csv"
VAL_TARGET_PATH = "../data/processed/target/y_val.csv"

In [10]:
# Load validation partitions
X_val = pd.read_csv(VAL_FEATURES_PATH)
y_val = pd.read_csv(VAL_TARGET_PATH).squeeze()

print(f"Validation features shape: {X_val.shape}")
print(f"Validation target distribution:\n{y_val.value_counts(normalize=True)}")

Validation features shape: (1600, 17)
Validation target distribution:
Exited
0    0.79625
1    0.20375
Name: proportion, dtype: float64


## 2. Query MLflow Registry for the Absolute Best Run

In [11]:
# Enforce experiment configuration check explicitly against our SQLite DB
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise RuntimeError(f"No active model runs found inside experiment: '{EXPERIMENT_NAME}'")

In [12]:
# Search for the best single run within this experiment using target metrics
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.pr_auc DESC", "metrics.test_auc DESC", "metrics.best_auc DESC"],
    max_results=1
)

if runs_df.empty:
    raise RuntimeError(f"No active model runs found inside experiment: '{EXPERIMENT_NAME}'")

In [13]:
best_run = runs_df.iloc[0]

In [14]:
print("=" * 60)
print(f"BEST PIPELINE FOUND IN SQLite DB")
print("=" * 60)
print(f"Model Flavor  : {best_run.get('tags.mlflow.runName', 'Unnamed Model')}")
print(f"MLflow Run ID : {best_run.run_id}")
print(f"Training Score: {best_run.get('metrics.pr_auc', best_run.get('metrics.best_auc', 0.0)):.4f} (PR-AUC)")
print("=" * 60)

BEST PIPELINE FOUND IN SQLite DB
Model Flavor  : optuna_search_parent
MLflow Run ID : 3a3946f4e23e4b45973ca0b28646548c
Training Score: 0.7063 (PR-AUC)


## 3. Load the Serialized Production Pipeline Model Object

In [15]:
# Construct direct model path URI using the run ID
model_uri = f"runs:/{best_run.run_id}/best_model"

try:
    # Try loading the optimized optuna model path first
    best_pipeline = mlflow.sklearn.load_model(model_uri)
except Exception:
    # Fallback path if utilizing standard baseline run wrappers
    model_uri = f"runs:/{best_run.run_id}/model"
    best_pipeline = mlflow.sklearn.load_model(model_uri)

print(f"Successfully unpacked model architecture state from: {model_uri}")

Successfully unpacked model architecture state from: runs:/3a3946f4e23e4b45973ca0b28646548c/best_model


## 4. Execute Generalization Inference on Validation Cohort

In [16]:
# Generate hard predictions and raw probability vectors
y_pred = best_pipeline.predict(X_val)
y_proba = best_pipeline.predict_proba(X_val)[:, 1]

## 5. Compute Generalization Performance Metrics

In [17]:
# Calculate precision, recall, and dual area under the curves
precision = precision_score(y_val, y_pred, zero_division=0)
recall = recall_score(y_val, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_val, y_proba)
pr_auc = average_precision_score(y_val, y_proba)

In [18]:
# Create a clean display frame
metrics_summary = pd.DataFrame({
    "Metric Name": ["Precision", "Recall (Sensitivity)", "ROC-AUC (Ranking)", "PR-AUC (Average Precision)"],
    "Validation Score": [precision, recall, roc_auc, pr_auc]
})

In [19]:
# Display clean tabular layout
print("\n### Final Validation Assessment Profile ###")
display(metrics_summary.style.format({"Validation Score": "{:.4f}"}))


### Final Validation Assessment Profile ###


,Metric Name,Validation Score
0,Precision,0.6290
1,Recall (Sensitivity),0.5460
2,ROC-AUC (Ranking),0.8521
3,PR-AUC (Average Precision),0.6717


### 5.1 Granular Classification Report Matrix

In [20]:
print("\n" + classification_report(y_val, y_pred, zero_division=0))


              precision    recall  f1-score   support

           0       0.89      0.92      0.90      1274
           1       0.63      0.55      0.58       326

    accuracy                           0.84      1600
   macro avg       0.76      0.73      0.74      1600
weighted avg       0.83      0.84      0.84      1600



### 5.2 Confusion Matrix Count Outputs

In [21]:
cm = confusion_matrix(y_val, y_pred)
cm_df = pd.DataFrame(
    cm, 
    index=['Actual Retained (0)', 'Actual Churned (1)'], 
    columns=['Predicted Retained (0)', 'Predicted Churned (1)']
)
display(cm_df)

,Predicted Retained (0),Predicted Churned (1)
Actual Retained (0),1169,105
Actual Churned (1),148,178
